# 00 · Validación previa en dos T4

Paquete experimental: encoder espectral congelado → dos cabezas afines → VAE.
Las corridas se distribuyen entre dos T4, una semilla por proceso. El eigensolve de cada corrida usa una GPU; no se suman las memorias de las dos tarjetas.

**Estado:** notebook preparado, sin resultados científicos precargados. La ejecución CUDA/SM75 debe validarse en Kaggle. Internet solo es necesario si falta una dependencia o para descargar MNIST.

Adjunta el ZIP o su contenido como un Kaggle Dataset y selecciona **GPU T4 x2**. El backend `flash` compila el kernel upstream incluido para SM75; `torch` es una alternativa explícita y no se debe reportar como FlashDiffusion. FlashDiffusionTests devolvió 404 durante la preparación: se fijó la fuente pública FlashDiffusion al commit indicado en README.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
ROOT = Path('/kaggle/working/dmap_vae_kaggle')
INPUT = Path('/kaggle/input')
CU = 'flash_diffusion.cu'
CODE = ['spectral.py', 'vae.py', 'data.py', 'experiment.py', 'launch.py', 'plots.py',
        'checks.py', 'prepare_alanine.py', 'generate_angles.py', 'SOURCE.json',
        'geometry.py', 'beta_sweep.py', 'config_beta_sweep.json']

# 1) Refresh the code on EVERY run (a partial copy left in /kaggle/working is never trusted).
#    The results in /kaggle/working/results_* are not touched.
sources = sorted({q.parent for q in INPUT.rglob('launch.py') if (q.parent / 'spectral.py').exists()})
archives = sorted(INPUT.rglob('dmap_vae_kaggle*.zip'))
if len(sources) > 1:
    raise FileNotFoundError(f'Hay varias copias del paquete en /kaggle/input: {sources}. Deja solo una.')
ROOT.mkdir(parents=True, exist_ok=True)
if sources:
    shutil.copytree(sources[0], ROOT, dirs_exist_ok=True)
elif len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall(ROOT.parent)
else:
    raise FileNotFoundError('Añade el paquete dmap_vae_kaggle como Kaggle Dataset; debe haber una sola copia.')
missing_code = [f for f in CODE if not (ROOT / f).exists()]
assert not missing_code, f'Faltan archivos del paquete: {missing_code}'

# 2) The CUDA source. Kaggle drops or flattens subfolders when a Dataset is built from loose
#    files, so look for it in vendor/, flattened at the root, or inside any attached ZIP.
target = ROOT / 'vendor' / CU
target.parent.mkdir(exist_ok=True)
if not target.exists():
    found = sorted(INPUT.rglob(CU))
    if found:
        shutil.copy2(found[0], target)
    else:
        for arc in sorted(INPUT.rglob('*.zip')):
            with zipfile.ZipFile(arc) as z:
                names = [n for n in z.namelist() if n.endswith('/' + CU) or n == CU]
                if names:
                    target.write_bytes(z.read(names[0]))
                    break
    lic = sorted(INPUT.rglob('LICENSE.FlashDiffusion'))
    if lic and not (target.parent / lic[0].name).exists():
        shutil.copy2(lic[0], target.parent / lic[0].name)
if not target.exists():
    raise FileNotFoundError(
        f'No se encontró {CU} en /kaggle/input. Sube la carpeta vendor/ (o el ZIP completo) '
        'al Kaggle Dataset; al subir archivos sueltos Kaggle omite las subcarpetas.')
expected = json.loads((ROOT / 'SOURCE.json').read_text())['sha256']
actual = hashlib.sha256(target.read_bytes()).hexdigest()
assert actual == expected, f'{CU} no coincide con SOURCE.json ({actual} != {expected}).'
print('Fuente CUDA verificada:', target)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
import importlib.util
missing = [m for m in ['numpy','scipy','pandas','matplotlib','ninja']
           if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import torch
assert torch.cuda.device_count() >= 2, 'Selecciona GPU T4 x2 en Kaggle.'
print([(i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i)) for i in range(2)])
(ROOT / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))

## Controles

Compara el kernel con referencia densa sobre las coordenadas efectivamente cuantizadas; valida ortogonalidad L²(π), consistencia Nyström, serialización, congelación del VAE y orientación angular. No constituye un benchmark científico ni certifica tasas de convergencia.

In [ ]:
# Compile the extension ONCE, serially and with visible output, before the two parallel checks
# (two processes compiling the same extension at once only produce interleaved, unreadable logs).
build = subprocess.run([sys.executable, '-c',
    'import spectral; print("Extensión compilada:", spectral.flash_extension())'],
    cwd=ROOT, env=dict(os.environ, CUDA_VISIBLE_DEVICES='0'), capture_output=True, text=True)
print(build.stdout[-4000:])
if build.returncode != 0:
    print(build.stderr[-8000:])
    raise RuntimeError('No continuar: falló la compilación de FlashDiffusion (ver log arriba).')

jobs = []
for gpu in (0, 1):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), OMP_NUM_THREADS='2')
    log = open(ROOT / f'checks_gpu{gpu}.log', 'w')
    jobs.append((gpu, log, subprocess.Popen([sys.executable, str(ROOT / 'checks.py'),
        '--backend', 'flash', '--device', 'cuda:0',
        '--out', str(ROOT / f'checks_gpu{gpu}.json')], env=env, stdout=log, stderr=subprocess.STDOUT)))
failed = []
for gpu, log, job in jobs:
    code = job.wait(); log.close()
    if code != 0:
        failed.append(gpu)
        print(f'--- GPU {gpu} falló (código {code}) ---')
        print((ROOT / f'checks_gpu{gpu}.log').read_text()[-6000:])
assert not failed, f'No continuar: falló un control numérico en GPU {failed}.'
for gpu in (0, 1):
    print((ROOT / f'checks_gpu{gpu}.json').read_text())